In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

# Breast Cancer Dataset — Exploratory Data Analysis

## 1. Load & Dataset Overview

In [ ]:
data = load_breast_cancer()

df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target
df["target_name"] = df["target"].map({0: "malignant", 1: "benign"})

print("Shape:", df.shape)
print("\nTarget classes:", data.target_names.tolist())
print("\nFeature names:")
for i, name in enumerate(data.feature_names):
    print(f"  {i+1:2d}. {name}")

In [ ]:
print("=== General Info ===")
df.info()

print("\n=== Missing Values ===")
print(df.isnull().sum().sum(), "total missing values")

print("\n=== Summary Statistics ===")
df.drop(columns=["target", "target_name"]).describe().T.round(2)

## 2. Class Distribution

In [ ]:
counts = df["target_name"].value_counts()
pcts = counts / counts.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
colors = ["#e74c3c", "#2ecc71"]
axes[0].bar(counts.index, counts.values, color=colors, edgecolor="white", width=0.5)
for i, (v, p) in enumerate(zip(counts.values, pcts.values)):
    axes[0].text(i, v + 3, f"{v}\n({p:.1f}%)", ha="center", fontsize=11)
axes[0].set_title("Class Counts")
axes[0].set_ylabel("Count")
axes[0].set_ylim(0, counts.max() * 1.2)

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct="%1.1f%%",
            colors=colors, startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 2})
axes[1].set_title("Class Proportions")

plt.suptitle("Target Class Distribution", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Per-Class Feature Statistics

In [ ]:
features = data.feature_names.tolist()

stats = df.groupby("target_name")[features].mean().T
stats.columns.name = None
stats["ratio_benign_malignant"] = (stats["benign"] / stats["malignant"]).round(2)
stats = stats.round(4)

print("Mean feature values by class (sorted by ratio):")
stats.sort_values("ratio_benign_malignant")

## 4. Feature Distributions by Class

In [ ]:
palette = {"malignant": "#e74c3c", "benign": "#2ecc71"}

n_cols = 5
n_rows = int(np.ceil(len(features) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3))
axes = axes.flatten()

for i, feat in enumerate(features):
    for label, color in palette.items():
        subset = df[df["target_name"] == label][feat]
        axes[i].hist(subset, bins=25, alpha=0.55, color=color, label=label, edgecolor="none")
    axes[i].set_title(feat, fontsize=8)
    axes[i].tick_params(labelsize=7)
    if i == 0:
        axes[i].legend(fontsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Feature Distributions — Malignant vs Benign", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 5. Boxplots per Class

In [ ]:
# Normalise features to [0,1] so all fit on the same axis
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df_norm = pd.DataFrame(scaler.fit_transform(df[features]), columns=features)
df_norm["target_name"] = df["target_name"].values

df_melt = df_norm.melt(id_vars="target_name", var_name="feature", value_name="value")

fig, ax = plt.subplots(figsize=(22, 6))
sns.boxplot(data=df_melt, x="feature", y="value", hue="target_name",
            palette=palette, width=0.6, linewidth=0.8, fliersize=2, ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.set_title("Normalised Feature Distributions by Class", fontsize=13, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Normalised Value")
ax.legend(title="Class")
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap

In [ ]:
corr = df[features].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(16, 13))
sns.heatmap(corr, mask=mask, annot=False, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.3, ax=ax, vmin=-1, vmax=1,
            cbar_kws={"shrink": 0.8})
ax.set_title("Feature Correlation Matrix", fontsize=14, fontweight="bold")
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.show()

# Top 10 most correlated pairs (excluding self)
corr_pairs = (corr.where(~mask)
              .stack()
              .reset_index()
              .rename(columns={"level_0": "feat_a", "level_1": "feat_b", 0: "correlation"}))
corr_pairs["abs_corr"] = corr_pairs["correlation"].abs()
print("Top 10 most correlated feature pairs:")
corr_pairs.sort_values("abs_corr", ascending=False).head(10)[["feat_a", "feat_b", "correlation"]]

## 7. Pairplot — Top Discriminative Features

In [ ]:
# Select top 6 features most correlated with the target (point-biserial via plain corr)
target_corr = df[features + ["target"]].corr()["target"].drop("target").abs()
top6 = target_corr.nlargest(6).index.tolist()

print("Top 6 features most correlated with target:")
for f in top6:
    print(f"  {f}: {target_corr[f]:.3f}")

pair_df = df[top6 + ["target_name"]]
g = sns.pairplot(pair_df, hue="target_name", palette=palette,
                 plot_kws={"alpha": 0.4, "s": 15},
                 diag_kind="kde")
g.figure.suptitle("Pairplot — Top 6 Discriminative Features", y=1.01, fontsize=13, fontweight="bold")
plt.show()

## 8. Feature Importance via Variance & Target Correlation

In [ ]:
target_corr_sorted = target_corr.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors_bar = ["#e74c3c" if v >= 0.5 else "#3498db" for v in target_corr_sorted.values]
bars = ax.barh(target_corr_sorted.index, target_corr_sorted.values, color=colors_bar, edgecolor="none")
ax.axvline(0.5, color="grey", linestyle="--", linewidth=1, label="|corr| = 0.5")
ax.set_xlabel("|Correlation with target|")
ax.set_title("Feature Correlation with Target (absolute)", fontsize=13, fontweight="bold")
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()